# Train a YOLO-pose keypoint detector for the FW-UAV (Phase 7c, part A)

Trains the smallest YOLO pose model on the FW-UAV6DPose train / val scenes prepared by `phase7c_prepare.py`
(13 keypoints per aircraft, 3967 train images from 85 scenes, every 2nd frame, and 331 val images from the 4 scenes of one
held-out background group; the **test scenes are not in the zip**). Same model and settings as Phase 7b; only the data changed.

**What you need to do**

1. Upload `data/fw_uav/yolo_fw_uav_7c.zip` (about 630 MB) to your Google Drive, into the folder **`MyDrive/formation-pose/`**.
2. In Colab: *Runtime -> Change runtime type -> T4 GPU*.
3. *Runtime -> Run all*. Allow access to Google Drive when asked. Training is 100 epochs at 1280 px (early stopping after
   20 epochs without improvement); there are about 2.5x more training images than in 7b, so expect an epoch to take about 2.5x as long as in 7b (an estimate, not measured). Keep the tab open.
4. If the session disconnects, open the notebook again and *Run all*: the training cell resumes from the last checkpoint
   saved on Drive.
5. When it finishes, download `MyDrive/formation-pose/runs/yolo_pose_fw_uav/weights/best.pt` and tell me; the evaluation on the test scenes comes next. Copy the printed validation metrics too.

In [ ]:
# 1) GPU check: stop early with a clear message if there is none
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Use Runtime -> Change runtime type -> T4 GPU, then run all cells again.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Mount Google Drive and unzip the dataset to /content (a fresh copy on every new session)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/formation-pose"
!ls -lh {DRIVE_DIR}/yolo_fw_uav_7c.zip
!unzip -q -o {DRIVE_DIR}/yolo_fw_uav_7c.zip -d /content
!ls /content/yolo_fw_uav_7c /content/yolo_fw_uav_7c/images/train | head -5
!cat /content/yolo_fw_uav_7c/data.yaml

In [ ]:
# 3) Install Ultralytics (YOLO)
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 4) Train. Checkpoints (last.pt every epoch, best.pt, a copy every 5 epochs) are written straight to Google Drive,
#    so nothing is lost if the session disconnects. Re-running this cell resumes from last.pt.
import os
from ultralytics import YOLO

DATA = "/content/yolo_fw_uav_7c/data.yaml"          # relative paths inside: images/train, images/val
RUNS = f"{DRIVE_DIR}/runs"
NAME = "yolo_pose_fw_uav_7c"       # new run name: must not resume the Phase 7b run on Drive
MODEL = "yolo11n-pose.pt"                         # the smallest YOLO pose model
last = f"{RUNS}/{NAME}/weights/last.pt"

if os.path.exists(last):
    print("Resuming from", last)
    YOLO(last).train(resume=True)
else:
    YOLO(MODEL).train(
        data=DATA,
        imgsz=1280,          # the aircraft is small in the 1920x1080 frames
        epochs=100,
        patience=20,         # early stopping on the validation split
        batch=8,             # fits a free T4 (16 GB) at 1280 px; raise to 16 if memory allows
        workers=2,
        device=0,
        project=RUNS, name=NAME, exist_ok=True,
        save_period=5,
        seed=0,
        fliplr=0.0,          # no horizontal flip: the wing-root keypoints are not exact mirror images (0.4-0.7 m off),
                             # so flipped labels would carry up to ~5 px of error. flip_idx stays in data.yaml.
    )

In [ ]:
# 5) Validation metrics of the best checkpoint (validation scenes only; the test scenes are not used here)
best = YOLO(f"{RUNS}/{NAME}/weights/best.pt")
m = best.val(data=DATA, imgsz=1280, batch=8, split="val")
print(f"Box  mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}")
print(f"Pose mAP50 {m.pose.map50:.3f}   mAP50-95 {m.pose.map:.3f}")
print("Best weights:", f"{RUNS}/{NAME}/weights/best.pt")